# 03 · 고객 세그먼트 분석 (군집화, 보조 분석)

**이탈 예측(02)과 독립적인 분석**이다. Target(Churn)을 학습에 쓰지 않고, 행동·이용 패턴만으로 고객을 나눈 뒤 사후적으로 군집별 이탈률을 비교해 검증한다.

산출물은  C 섹션과 (선택) Streamlit 유지 전략 문구에 반영한다.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.predict import load_config, resolve_path
from src.clean import clean_raw  # 공통 정제 규칙 (A 관리)

cfg = load_config()
df = clean_raw(pd.read_csv(resolve_path(cfg['data']['raw_path']), encoding=cfg['data']['encoding'], sep=cfg['data']['sep']))
tcol = cfg['target']['column']; pos = str(cfg['target']['positive_label']).strip().lower()
y = df[tcol].astype(str).str.strip().str.lower().eq(pos).astype(int)  # 검증에만 사용, 학습엔 미사용

## 1) 군집용 Feature 선택
행동·이용 패턴 위주로 선택한다. ID·Target·리텐션 컬럼(순환 논리 소지)은 제외.

In [ ]:
cluster_cols = ['MonthlyRevenue','MonthlyMinutes','CurrentEquipmentDays','MonthsInService',
                'OverageMinutes','DroppedCalls','CustomerCareCalls','PercChangeMinutes']  # TODO 팀 판단으로 조정
X = df[cluster_cols].apply(pd.to_numeric, errors='coerce')
X = X.fillna(X.median())

## 2) 스케일링 + 군집 수 결정 (Elbow / Silhouette)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

Xs = StandardScaler().fit_transform(X)

inertias, sils = [], []
ks = range(2, 8)
for k in ks:
    km = KMeans(n_clusters=k, random_state=cfg['project']['random_state'], n_init=10).fit(Xs)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(Xs, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(list(ks), inertias, marker='o'); ax[0].set_title('Elbow (inertia)')
ax[1].plot(list(ks), sils, marker='o'); ax[1].set_title('Silhouette score')
plt.show()

## 3) 최종 군집화 + 프로필

In [ ]:
K = 4  # TODO: 위 그래프 근거로 결정
km = KMeans(n_clusters=K, random_state=cfg['project']['random_state'], n_init=10).fit(Xs)
df['cluster'] = km.labels_

profile = df.groupby('cluster')[cluster_cols].mean().round(1)
profile['n_customers'] = df.groupby('cluster').size()
profile

## 4) 군집별 이탈률 (사후 검증)
군집이 이탈과 무관하게 갈리면(모든 군집이 비슷한 이탈률) 세그먼트로서 의미가 약하다는 뜻 — 한계로 기록.

In [ ]:
df.groupby('cluster').apply(lambda g: y.loc[g.index].mean(), include_groups=False).rename('churn_rate').mul(100).round(1)

## 5) 정리 → reports/report.md C 섹션
- 군집별 특징 요약, 유지 전략 제안, 한계